In [ ]:

import subprocess
subprocess.run(["pip", "install", "ultralytics", "-q"], check=True)
from ultralytics import YOLO
import shutil, random, yaml
from pathlib import Path
from collections import defaultdict
import pandas as pd
random.seed(42)
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image
import random


In [ ]:

# ════════════════════════════════════════════════════════════════
# 1. PATHS
# ════════════════════════════════════════════════════════════════
HUMAN_ROOT = Path("/kaggle/input/datasets/abdelwahabselim2026/dataaaa/HumanDataset")
SD_ROOT    = Path("/kaggle/input/datasets/abdelwahabselim2026/dataaaa/Self Driving")
SIGNS_ROOT = Path("/kaggle/input/datasets/pkdarabi/cardetection/car")
SPD_ROOT   = Path("/kaggle/input/datasets/abdelwahabselim2026/data333")
TL_ROOT    = Path("/kaggle/input/datasets/abdelwahabselim2026/data4444")
OUT_ROOT   = Path("/kaggle/working/yolo_dataset_v5")

SPLITS      = ["train", "valid", "test"]
SPLIT_RATIO = (0.80, 0.10, 0.10)

# ════════════════════════════════════════════════════════════════
# 2. FINAL CLASSES
# ════════════════════════════════════════════════════════════════
FINAL_CLASSES = [
    "person",             # 0
    "car",                # 1
    "traffic light red",  # 2
    "traffic light green",# 3
    "speed limit 30",     # 4
    "speed limit 70",     # 5
    "speed limit 100",    # 6
    "stop sign",          # 7
]

# ════════════════════════════════════════════════════════════════
# 3. CLASS MAPS
# ════════════════════════════════════════════════════════════════
# HumanDataset: [0]"1" [1]Human [2]Pedestrian [3]Person [4]child [5]human [6]person
HUMAN_MAP = {0:0, 1:0, 2:0, 3:0, 4:0, 5:0, 6:0}

# Self Driving: [1]car [2]pedestrian [4]tL-Green [5]tL-GreenLeft [6]tL-Red [7]tL-RedLeft
SD_MAP = {1:1, 2:0, 4:3, 5:3, 6:2, 7:2}

# Traffic Signs: [0]GreenLight [1]RedLight [3]SL100 [7]SL30 [11]SL70 [14]Stop
SIGNS_MAP = {0:3, 1:2, 3:6, 7:4, 11:5, 14:7}

# data333: [0]SL100 [3]SL30 [5]SL70 [8]Stop
SPD_MAP = {0:6, 3:4, 5:5, 8:7}

# data4444: [0]green [1]red [2]yellow(skip)
TL_MAP = {0:3, 1:2}

# ════════════════════════════════════════════════════════════════
# 4. CAPS per split
# ════════════════════════════════════════════════════════════════
#              person   car   tl-red  tl-green
# train    →   8,000  6,400   4,000    3,200
# valid    →   1,000    800     500      400
# test     →   1,000    800     500      400
CAPS = {
    "train": {0: 8_000, 1: 6_400, 2: 4_000, 3: 3_200},
    "valid": {0: 1_000, 1:   800, 2:   500, 3:   400},
    "test":  {0: 1_000, 1:   800, 2:   500, 3:   400},
}

# ════════════════════════════════════════════════════════════════
# 5. HELPERS
# ════════════════════════════════════════════════════════════════
ann_budget = {"train": defaultdict(int),
              "valid": defaultdict(int),
              "test":  defaultdict(int)}
stats      = defaultdict(lambda: defaultdict(int))

def setup_dirs(root):
    for sp in SPLITS:
        (root / sp / "images").mkdir(parents=True, exist_ok=True)
        (root / sp / "labels").mkdir(parents=True, exist_ok=True)
    print(f"✅ Output dirs: {root}")

def find_img(lbl, img_dir):
    for ext in [".jpg", ".jpeg", ".png", ".bmp"]:
        p = img_dir / (lbl.stem + ext)
        if p.exists(): return p
    return None

def remap(txt_path, cls_map):
    out = []
    for line in open(txt_path):
        line = line.strip()
        if not line: continue
        parts = line.split()
        cid = int(parts[0])
        if cid in cls_map:
            out.append(f"{cls_map[cid]} {' '.join(parts[1:])}")
    return out

def budget_ok(lines, sp):
    tmp = defaultdict(int)
    for line in lines:
        tmp[int(line.split()[0])] += 1
    return all(
        ann_budget[sp][cid] + cnt <= CAPS[sp][cid]
        for cid, cnt in tmp.items() if cid in CAPS[sp]
    )

def budget_add(lines, sp):
    for line in lines:
        ann_budget[sp][int(line.split()[0])] += 1

def write_pair(img_src, lines, out_img, out_lbl):
    shutil.copy2(img_src, out_img)
    with open(out_lbl, "w") as f:
        f.write("\n".join(lines))

def track(lines, sp):
    for line in lines:
        stats[sp][FINAL_CLASSES[int(line.split()[0])]] += 1

def process_presplit(tag, root, cls_map, prefix="", use_budget=True):
    print(f"\n{'═'*60}")
    print(f"  {tag}")
    print(f"{'═'*60}")
    for sp in SPLITS:
        lbl_dir = root / sp / "labels"
        img_dir = root / sp / "images"
        if not lbl_dir.exists():
            print(f"  {sp:<6}: ⚠️  not found"); continue
        ok = skip = 0
        for txt in sorted(lbl_dir.glob("*.txt")):
            img = find_img(txt, img_dir)
            if not img: skip += 1; continue
            lines = remap(txt, cls_map)
            if not lines: skip += 1; continue
            if use_budget and not budget_ok(lines, sp): skip += 1; continue
            pf = f"{prefix}_" if prefix else ""
            write_pair(img, lines,
                       OUT_ROOT/sp/"images"/(pf+img.name),
                       OUT_ROOT/sp/"labels"/(pf+txt.name))
            if use_budget: budget_add(lines, sp)
            track(lines, sp); ok += 1
        print(f"  {sp:<6}: ✅ {ok:>6,}  ⏭️  {skip:>5,}")

def print_budget(cls_ids, names):
    for cid in cls_ids:
        tr = ann_budget["train"][cid]
        va = ann_budget["valid"][cid]
        te = ann_budget["test"][cid]
        print(f"  {names[cid]:<22}: train={tr:,}  valid={va:,}  test={te:,}")


# ════════════════════════════════════════════════════════════════
# 6. BUILD DATASET
# ════════════════════════════════════════════════════════════════
setup_dirs(OUT_ROOT)

# ── [1/5] HumanDataset ──────────────────────────────────────────
process_presplit("[1/5] HumanDataset → person",
                 HUMAN_ROOT, HUMAN_MAP, prefix="", use_budget=True)
print_budget([0], FINAL_CLASSES)

# ── [2/5] Traffic Signs ─────────────────────────────────────────
process_presplit("[2/5] Traffic Signs → tl-red/green, SL30/70/100, stop",
                 SIGNS_ROOT, SIGNS_MAP, prefix="SGN", use_budget=True)
print_budget([2, 3], FINAL_CLASSES)

# ── [3/5] data333 ───────────────────────────────────────────────
process_presplit("[3/5] data333 → SL30, SL70, SL100, stop",
                 SPD_ROOT, SPD_MAP, prefix="SPD", use_budget=False)
print(f"  SL30 : {sum(stats[sp].get('speed limit 30',0)  for sp in SPLITS):,}")
print(f"  SL70 : {sum(stats[sp].get('speed limit 70',0)  for sp in SPLITS):,}")
print(f"  SL100: {sum(stats[sp].get('speed limit 100',0) for sp in SPLITS):,}")
print(f"  stop : {sum(stats[sp].get('stop sign',0)       for sp in SPLITS):,}")

# ── [4/5] data4444 (traffic lights, no pre-split) ───────────────
print(f"\n{'═'*60}")
print(f"  [4/5] data4444 → tl-red, tl-green")
print(f"{'═'*60}")

# data4444 has train/valid/test but valid is only 50 images → treat as no-split
# and redistribute ourselves for better balance
TL_LBL_ALL = []
TL_IMG_ALL = []
for sp in SPLITS:
    lbl_dir = TL_ROOT / sp / "labels"
    img_dir = TL_ROOT / sp / "images"
    if not lbl_dir.exists(): continue
    for txt in sorted(lbl_dir.glob("*.txt")):
        img = find_img(txt, img_dir)
        if not img: continue
        lines = remap(txt, TL_MAP)
        if lines:
            TL_LBL_ALL.append((txt, img, lines))

random.shuffle(TL_LBL_ALL)
n = len(TL_LBL_ALL)
a, b = int(n*0.80), int(n*0.10)
tl_splits = {
    "train": TL_LBL_ALL[:a],
    "valid": TL_LBL_ALL[a:a+b],
    "test":  TL_LBL_ALL[a+b:],
}
print(f"  Total usable: {n:,}  → train:{a:,} valid:{b:,} test:{n-a-b:,}")

for sp, items in tl_splits.items():
    ok = skip = 0
    for txt, img, lines in items:
        if not budget_ok(lines, sp): skip += 1; continue
        write_pair(img, lines,
                   OUT_ROOT/sp/"images"/("TL_"+img.name),
                   OUT_ROOT/sp/"labels"/("TL_"+txt.name))
        budget_add(lines, sp); track(lines, sp); ok += 1
    print(f"  {sp:<6}: ✅ {ok:>6,}  ⏭️  {skip:>5,}")
print_budget([2, 3], FINAL_CLASSES)

# ── [5/5] Self Driving (car capped, no pre-split) ───────────────
print(f"\n{'═'*60}")
print(f"  [5/5] Self Driving → car (≤8,000), person, tl-red/green")
print(f"{'═'*60}")
SD_LBL = SD_ROOT / "export" / "labels"
SD_IMG  = SD_ROOT / "export" / "images"

sd_valid    = [t for t in sorted(SD_LBL.glob("*.txt")) if remap(t, SD_MAP)]
car_files   = [t for t in sd_valid if any(int(l.split()[0])==1 for l in remap(t,SD_MAP))]
other_files = [t for t in sd_valid if t not in set(car_files)]
random.shuffle(car_files)
sd_files = car_files + other_files
print(f"  usable: {len(sd_valid):,}  |  car: {len(car_files):,}  |  other: {len(other_files):,}")

random.shuffle(sd_files)
n = len(sd_files)
a, b = int(n*SPLIT_RATIO[0]), int(n*SPLIT_RATIO[1])
for sp, files in [("train",sd_files[:a]),("valid",sd_files[a:a+b]),("test",sd_files[a+b:])]:
    ok = skip = 0
    for txt in files:
        img = find_img(txt, SD_IMG)
        if not img: skip += 1; continue
        lines = remap(txt, SD_MAP)
        if not lines or not budget_ok(lines, sp): skip += 1; continue
        write_pair(img, lines,
                   OUT_ROOT/sp/"images"/("SD_"+img.name),
                   OUT_ROOT/sp/"labels"/("SD_"+txt.name))
        budget_add(lines, sp); track(lines, sp); ok += 1
    print(f"  {sp:<6}: ✅ {ok:>6,}  ⏭️  {skip:>5,}")
print_budget([0, 1, 2, 3], FINAL_CLASSES)


# ════════════════════════════════════════════════════════════════
# 7. WRITE data.yaml
# ════════════════════════════════════════════════════════════════
yaml_path = OUT_ROOT / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump({
        "path" : str(OUT_ROOT),
        "train": "train/images",
        "val"  : "valid/images",
        "test" : "test/images",
        "nc"   : len(FINAL_CLASSES),
        "names": FINAL_CLASSES,
    }, f, default_flow_style=False, allow_unicode=True)
print(f"\n✅ data.yaml → {yaml_path}")


# ════════════════════════════════════════════════════════════════
# 8. FINAL STATS
# ════════════════════════════════════════════════════════════════
print("\n" + "═"*70)
print("  📊  FINAL DATASET STATS")
print("═"*70)
for sp in SPLITS:
    n = len(list((OUT_ROOT/sp/"images").glob("*")))
    print(f"  {sp:<6}: {n:>7,} images")

rows = []
for cls in FINAL_CLASSES:
    row = {"Class": cls}
    for sp in SPLITS:
        row[sp.capitalize()] = stats[sp].get(cls, 0)
    row["Total"] = sum(stats[sp].get(cls,0) for sp in SPLITS)
    rows.append(row)
df = pd.DataFrame(rows)
print("\n" + df.to_string(index=False))
totals = [r["Total"] for r in rows]
print(f"\n  Max: {max(totals):,}  |  Min: {min(totals):,}  |  Ratio: {max(totals)/max(min(totals),1):.1f}x")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 9. TRAIN YOLOv8n
# ════════════════════════════════════════════════════════════════
model = YOLO("yolov8n.pt")
model.train(
    data         = str(yaml_path),
    epochs       = 100,
    imgsz        = 128,
    batch        = 16,
    name         = "yolov8n_v5",
    project      = "/kaggle/working/runs",
    optimizer    = "AdamW",
    lr0          = 0.001,
    lrf          = 0.01,
    weight_decay = 0.0005,
    momentum     = 0.937,
    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.4,
    degrees      = 5.0,
    translate    = 0.1,
    scale        = 0.5,
    fliplr       = 0.5,
    mosaic       = 1.0,
    mixup        = 0.1,
    copy_paste   = 0.1,
    workers      = 2,
    cache        = False,
    amp          = True,
    patience     = 15,
    save         = True,
    verbose      = True,
)
print("\n✅ Training done!")
print("   Best → /kaggle/working/runs/yolov8n_v5/weights/best.pt")


# ════════════════════════════════════════════════════════════════
# 10. TEST EVALUATION
# ════════════════════════════════════════════════════════════════
print("\n" + "═"*70)
print("  🧪  TEST SET EVALUATION")
print("═"*70)

best = YOLO("/kaggle/working/runs/yolov8n_v5/weights/best.pt")
m    = best.val(
    data    = str(yaml_path),
    split   = "test",
    imgsz   = 128,
    batch   = 16,
    name    = "test_eval_v8",
    project = "/kaggle/working/runs",
)

print(f"\n  mAP50    : {m.box.map50:.4f}")
print(f"  mAP50-95 : {m.box.map:.4f}")
print(f"  Precision: {m.box.mp:.4f}")
print(f"  Recall   : {m.box.mr:.4f}")
print(f"\n  Per-class AP50:")
for i, cls in enumerate(FINAL_CLASSES):
    ap  = m.box.ap50[i] if i < len(m.box.ap50) else 0
    bar = "█" * int(ap * 20)
    print(f"    {cls:<22}: {ap:.4f}  {bar}")

print("\n🏁 Pipeline complete!")

In [ ]:
print("\n" + "═"*70)
print("  🧪  TEST SET EVALUATION")
print("═"*70)

best = YOLO("/kaggle/working/runs/yolov8n_v5/weights/best.pt")
m    = best.val(
    data    = str(yaml_path),
    split   = "test",
    imgsz   = 128,
    batch   = 16,
    name    = "test_eval_v8",
    project = "/kaggle/working/runs",
)

print(f"\n  mAP50    : {m.box.map50:.4f}")
print(f"  mAP50-95 : {m.box.map:.4f}")
print(f"  Precision: {m.box.mp:.4f}")
print(f"  Recall   : {m.box.mr:.4f}")
print(f"\n  Per-class AP50:")
for i, cls in enumerate(FINAL_CLASSES):
    ap  = m.box.ap50[i] if i < len(m.box.ap50) else 0
    bar = "█" * int(ap * 20)
    print(f"    {cls:<22}: {ap:.4f}  {bar}")

print("\n🏁 Pipeline complete!")

In [ ]:

# ── CONFIG ───────────────────────────────────────────────────────
MODEL_PATH  = "/kaggle/working/runs/yolov8n_v5/weights/best.pt""
DATASET_DIR = Path("/kaggle/working/yolo_dataset_v5")
OUT_PATH    = "/kaggle/working/Fig_X85_One_Per_Class.png"
IMGSZ       = 128
CONF        = 0.40

COLORS = {
    "person"             : "#E53935",
    "car"                : "#1E88E5",
    "traffic light red"  : "#D81B60",
    "traffic light green": "#43A047",
    "speed limit 30"     : "#FB8C00",
    "speed limit 70"     : "#E65100",
    "speed limit 100"    : "#BF360C",
    "stop sign"          : "#8E24AA",
}
DEFAULT_COLOR = "#2F5496"

# ── LOAD MODEL ───────────────────────────────────────────────────
model      = YOLO(MODEL_PATH)
class_names = model.names  # {0: 'person', 1: 'car', ...}
print(f"Classes: {class_names}")

# ── FIND ALL IMAGES ──────────────────────────────────────────────
all_imgs = []
for split in ["test", "valid", "val", "train"]:
    d = DATASET_DIR / split / "images"
    if d.exists():
        found = list(d.glob("*.jpg")) + list(d.glob("*.png"))
        all_imgs.extend(found)
        print(f"  {split}: {len(found)} images")

print(f"Total images: {len(all_imgs)}")
# random.seed(42)
random.shuffle(all_imgs)

# ── SCAN IMAGES UNTIL WE HAVE ONE PER CLASS ──────────────────────
target_classes = set(class_names.values())
class_to_result = {}   # class_name → (img_path, result, box)

print(f"\nSearching for one image per class...")
print(f"Target classes: {target_classes}\n")

BATCH = 32   

for start in range(0, min(len(all_imgs), 3000), BATCH):
    if set(class_to_result.keys()) == target_classes:
        break   

    batch_paths = all_imgs[start : start + BATCH]
    results = model.predict(
        source=[str(p) for p in batch_paths],
        imgsz=IMGSZ, conf=CONF,
        verbose=False, device="cpu", save=False,
    )

    for img_path, r in zip(batch_paths, results):
        if r.boxes is None or len(r.boxes) == 0:
            continue
        for box in r.boxes:
            cls_id   = int(box.cls[0])
            cls_name = class_names.get(cls_id, str(cls_id))
            conf_val = float(box.conf[0])
            if cls_name not in class_to_result:
                class_to_result[cls_name] = (img_path, r)
                print(f"  ✅ Found: {cls_name:<25} conf={conf_val:.2f}  "
                      f"({img_path.name})")
            if set(class_to_result.keys()) == target_classes:
                break
        if set(class_to_result.keys()) == target_classes:
            break

print(f"\nFound {len(class_to_result)}/{len(target_classes)} classes")
missing = target_classes - set(class_to_result.keys())
if missing:
    print(f"⚠ Missing classes: {missing}")
    print("  Try lowering CONF to 0.10 or including 'train' split")


ordered_classes = [
    "person", "car",
    "traffic light red", "traffic light green",
    "speed limit 30", "speed limit 70", "speed limit 100",
    "stop sign",
]

for c in class_to_result:
    if c not in ordered_classes:
        ordered_classes.append(c)

found_classes = [c for c in ordered_classes if c in class_to_result]

cols = 4
rows = (len(found_classes) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols,
                          figsize=(cols * 4.5, rows * 4.2))
fig.patch.set_facecolor("white")
axes = np.array(axes).flatten()

for i, cls_name in enumerate(found_classes):
    ax        = axes[i]
    img_path, r = class_to_result[cls_name]
    color     = COLORS.get(cls_name, DEFAULT_COLOR)

    img = np.array(Image.open(img_path).convert("RGB"))
    display_h, display_w = img.shape[:2]

    ax.imshow(img, interpolation="bicubic")
    ax.set_xlim(0, display_w)
    ax.set_ylim(display_h, 0)

    scale_x = display_w / r.orig_shape[1]
    scale_y = display_h / r.orig_shape[0]

    best_conf = 0
    for box in r.boxes:
        bid      = int(box.cls[0])
        bname    = class_names.get(bid, str(bid))
        conf_val = float(box.conf[0])
        bcolor   = COLORS.get(bname, DEFAULT_COLOR)

        x1, y1, x2, y2 = box.xyxy[0].tolist()
        x1 *= scale_x; x2 *= scale_x
        y1 *= scale_y; y2 *= scale_y

        is_target = (bname == cls_name)
        lw        = 2.5 if is_target else 1.0
        alpha     = 0.95 if is_target else 0.45

        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=lw, edgecolor=bcolor,
            facecolor="none", zorder=3, alpha=alpha
        )
        ax.add_patch(rect)

        if is_target and conf_val > best_conf:
            best_conf = conf_val
            # label على أفضل detection
            ax.text(
                x1, max(y1 - 4, 0),
                f"{cls_name}  {conf_val:.2f}",
                fontsize=7.5, color="white", fontweight="bold",
                bbox=dict(boxstyle="square,pad=1.3",
                          facecolor=color, edgecolor="none",
                          alpha=0.92),
                zorder=5
            )

    ax.set_title(
        cls_name.upper(),
        fontsize=9.5, fontweight="bold",
        color=color, pad=5
    )
    ax.axis("off")


for j in range(len(found_classes), len(axes)):
    axes[j].set_visible(False)
handles = [
    patches.Patch(facecolor=COLORS.get(c, DEFAULT_COLOR),
                  edgecolor="none", label=c)
    for c in found_classes
]
fig.legend(
    handles=handles, loc="lower center",
    ncol=4, fontsize=9, framealpha=0.9,
    edgecolor="#ccc", bbox_to_anchor=(0.5, -0.03)
)

fig.suptitle(
    "Figure X.8.5  –  One Sample Detection per Class  "
    "|  YOLOv8n  |  128 × 128 px",
    fontsize=13, color="#1F3864",
    fontweight="bold", y=1.02
)
fig.text(
    0.5, -0.07,
    "Each panel shows a representative detection for one of the eight target classes.\n"
    "Thick bounding box = target class. Thin boxes = other detected objects in the same frame.",
    ha="center", fontsize=8.5, color="#555", style="italic"
)

plt.tight_layout(h_pad=2.0, w_pad=1.2)
plt.savefig(OUT_PATH, dpi=200, bbox_inches="tight", facecolor="white")
plt.show()
print(f"\n✅ Saved: {OUT_PATH}")